# Notebook 14 (Revised) — Frozen LightGBM final refits (separately gated)

**Status:** Read-only final-refit preflight executed and verified; no final models fitted. This is a **new notebook**, not a modification of historical `14_pvod_external_validation.ipynb`.

This notebook follows the immutable development-CV model selections locked in Git commit **`c6a9162`** and the completed read-only Notebook 13 (`390ad2f`). It can reconstruct the three eligible development-only training populations and, **only after a separate deliberate opt-in**, fit **one frozen LightGBM estimator per horizon**. It contains **no terminal forecasting, predictions, scoring, hyperparameter search, or model comparison on test data**.

| Horizon | Model | Selected family | Frozen configuration | Expected training rows |
|:---:|---|---|---|---:|
| 15 min | LightGBM C1 | F1 | 300 estimators, LR 0.03, 15 leaves | 104,817 |
| 30 min | LightGBM C1 | F2 | same | 104,814 |
| 60 min | LightGBM C1 | F2 | same | 104,808 |

**Safe default:** `RUN_FINAL_REFIT = False` and `REFIT_HORIZONS = ()`. Restart Kernel → Run All performs audits, input rebuilding, and **zero fits/writes**. After reviewing that preflight, an operator can *separately authorize a single final refit* by setting the three explicit controls in Section 1, one horizon at a time. The saved research version must revert to the safe defaults. Model outputs, if explicitly authorized, go to **`results/final_model_refits/revised_lightgbm_v1/`**, never into frozen `results/development_only/`.

**Boundary policy:** Full HDF5 timestamps and column names may be inspected as metadata only. Measurement cells must be loaded **only before 2019-05-31 23:00 UTC**. The raw September 1 boundary point is not a training observation. Historical June–August 2019 results were inspected by older research; none of those observations is needed or accessed here.

## 1. Resource and authorization controls

These flags are intentionally **disabled** in the saved notebook. The explicit token plus exactly **one** requested horizon are additional defenses against accidental `Run All` training. Changing these flags does not authorize terminal evaluation; this notebook has no such execution path.

In [1]:
from __future__ import annotations

from pathlib import Path
from IPython.display import display
import hashlib
import importlib.metadata as metadata
import json
import os
import platform
import subprocess
import sys
import time

import numpy as np
import pandas as pd

# -------- SAFE, REVIEWABLE DEFAULTS: NEVER COMMIT WITH TRAINING ENABLED --------
RUN_FINAL_REFIT = False
REFIT_HORIZONS = ()  # When explicitly authorized, choose ONE of (15,), (30,), (60,).
REFIT_AUTHORIZATION_TOKEN = ""  # Required exact text: AUTHORIZE_FROZEN_DEVELOPMENT_REFIT
TERMINAL_EVALUATION_AUTHORIZED = False  # ALWAYS False; no terminal code exists.
# -------------------------------------------------------------------------------

assert TERMINAL_EVALUATION_AUTHORIZED is False
if RUN_FINAL_REFIT:
    assert REFIT_HORIZONS in ((15,), (30,), (60,)), "One horizon per authorized run"
    assert REFIT_AUTHORIZATION_TOKEN == "AUTHORIZE_FROZEN_DEVELOPMENT_REFIT", (
        "Explicit independent authorization is required before any model fitting"
    )
else:
    assert REFIT_HORIZONS == () and REFIT_AUTHORIZATION_TOKEN == "", (
        "Disable all refit controls for a safe read-only Run All"
    )

ROOT_CANDIDATES = []
if os.environ.get("SOLETE_PROJECT_ROOT"):
    ROOT_CANDIDATES.append(Path(os.environ["SOLETE_PROJECT_ROOT"]).expanduser())
ROOT_CANDIDATES.extend([Path.cwd(), *Path.cwd().parents])
PROJECT_ROOT = next((p.resolve() for p in ROOT_CANDIDATES
                     if (p / ".git").exists()
                     and (p / "scripts/verify_revised_selection_lock.py").is_file()
                     and (p / "notebooks/13_revised_final_refit_preflight.ipynb").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run in the SOLETE repository root or set SOLETE_PROJECT_ROOT")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("RUN FINAL REFIT:", "EXPLICITLY OPTED IN" if RUN_FINAL_REFIT else "DISABLED")
print("TERMINAL EVALUATION: DISABLED (no implementation)")
print("Repository:", PROJECT_ROOT)

RUN FINAL REFIT: DISABLED
TERMINAL EVALUATION: DISABLED (no implementation)
Repository: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project


## 2. Verify immutable selection lock, source, and runtime

Require the exact bytes of the selection lock as recorded at `c6a9162`. Invoke the read-only verifier instead of reproducing its complex archive logic here. Validate Python and frozen model dependencies. A mismatch is fatal even if `RUN_FINAL_REFIT` is disabled.

In [2]:
LOCK_COMMIT = "c6a9162"
NOTEBOOK_13_COMMIT = "390ad2f"
LOCK_REL = "results/development_only/revised_model_selection_lock_v1.json"
lock_path = PROJECT_ROOT / LOCK_REL
lock_bytes = lock_path.read_bytes()
lock = json.loads(lock_bytes)

def git(*args, check=True, text=True):
    return subprocess.run(["git", *args], cwd=PROJECT_ROOT,
                          check=check, capture_output=True, text=text)

assert git("merge-base", "--is-ancestor", LOCK_COMMIT, "HEAD").returncode == 0
assert git("merge-base", "--is-ancestor", NOTEBOOK_13_COMMIT, "HEAD").returncode == 0
assert lock_bytes == git("show", f"{LOCK_COMMIT}:{LOCK_REL}", text=False).stdout, (
    "Selection lock differs from committed immutable source"
)
assert lock["freeze_state"] == "PROPOSED_LOCK_NOT_YET_COMMITTED", (
    "Preparation-time label remains in committed lock: commit is the effective freeze"
)
assert lock["evaluation_policy"]["terminal_access_authorized"] is False
assert lock["evaluation_policy"]["extra_training_authorized"] is False
assert lock["refit_protocol"]["phase"] == "DESIGN_FROZEN_BUT_REFIT_NOT_PERFORMED"
assert lock["refit_protocol"]["do_not_fit_or_score_terminal_data"] is True
assert lock["archived_audit"]["archived_fold_scores"] == 1665
lock_sha256 = hashlib.sha256(lock_bytes).hexdigest()

verification = subprocess.run([sys.executable, "scripts/verify_revised_selection_lock.py"],
                              cwd=PROJECT_ROOT, text=True, capture_output=True, check=True)
assert "MODEL-SELECTION LOCK PREFLIGHT: PASSED" in verification.stdout
assert "1,665 / 1,665" in verification.stdout
assert "NO MODEL TRAINING / NO TEST DATA ACCESS" in verification.stdout
print(verification.stdout)

assert sys.version_info[:2] == (3, 12)
for package, expected in lock["reproducibility"]["dependencies_pinned_from_pyproject_toml"].items():
    actual = metadata.version(package)
    assert actual == expected, f"Package mismatch: {package}: expected {expected}, got {actual}"
    print(f"{package}: {actual}")

from src.classical_models import (
    LIGHTGBM_CONFIGS, LIGHTGBM_FIXED_PARAMS, build_lightgbm_model,
)
selections = {int(c["horizon_minutes"]): c for c in lock["selected_candidates"]}
assert set(selections) == {15, 30, 60}
assert [selections[h]["feature_family"] for h in (15, 30, 60)] == ["F1", "F2", "F2"]
assert LIGHTGBM_FIXED_PARAMS == lock["refit_protocol"]["lightgbm_fixed_parameters_from_frozen_source"]
for h in (15, 30, 60):
    c = selections[h]
    assert c["model"] == "LightGBM" and c["candidate_id"] == "LightGBM_C1"
    assert c["configuration_index_zero_based"] == 0
    assert c["configuration"] == dict(LIGHTGBM_CONFIGS[0])
    assert c["configuration"] == {"n_estimators": 300, "learning_rate": 0.03, "num_leaves": 15}
    assert c["refit"]["early_stopping"] is False
    model = build_lightgbm_model(c["configuration"])  # Constructor; NOT fitted.
    assert not hasattr(model, "booster_")
    del model
print("LOCK / SOURCE / PACKAGE / UNFITTED MODEL SPECIFICATION: PASSED")

15 min: LightGBM F1 C1  MAE=0.177400 kW  RF challenger MAE=0.177618 kW
30 min: LightGBM F2 C1  MAE=0.219329 kW  RF challenger MAE=0.225092 kW
60 min: LightGBM F2 C1  MAE=0.280951 kW  RF challenger MAE=0.286863 kW
MODEL-SELECTION LOCK PREFLIGHT: PASSED — source and archive evidence consistent
1,665 / 1,665 candidate-fold evaluations source/identity audited
NO MODEL TRAINING / NO TEST DATA ACCESS / NO OUTPUT FILE WRITES

lightgbm: 4.7.0
numpy: 2.0.2
pandas: 3.0.5
scikit-learn: 1.9.0
LOCK / SOURCE / PACKAGE / UNFITTED MODEL SPECIFICATION: PASSED


## 3. Validate time boundaries from **HDF5 metadata only**

The raw five-minute file includes a final September 1 boundary point (**131,617 timestamps**). No HDF5 measurement cells are accessed in this section. Final refit ends at the exclusive **May 31, 2019, 23:00 UTC** label-availability cutoff.

In [3]:
import h5py
from src.forecasting_protocol import (
    DAILY_LAG_STEPS, F0_HISTORY_SOURCES, FORECAST_HORIZONS,
    EXPECTED_TERMINAL_REFIT_COUNTS_BY_HORIZON,
    MODEL_AVAILABILITY_GAP, TARGET_COLUMN, TEST_START, TRAIN_START,
    make_terminal_refit_mask,
)

H5_PATH = PROJECT_ROOT / "solete_dataset/SOLETE_Pombo_5min.h5"
assert H5_PATH.is_file(), f"Missing local five-minute SOLETE HDF5: {H5_PATH}"
CUTOFF = pd.Timestamp(lock["refit_protocol"]["development_valid_time_cutoff_utc_exclusive"])
assert CUTOFF == pd.Timestamp("2019-05-31 23:00:00", tz="UTC")
assert CUTOFF == TEST_START - MODEL_AVAILABILITY_GAP
assert tuple(FORECAST_HORIZONS) == (15, 30, 60)
assert DAILY_LAG_STEPS == 288

with h5py.File(H5_PATH, "r") as f:
    ds = f["DATA"]
    full_index = pd.DatetimeIndex(pd.to_datetime(ds["axis1"][:], unit="ns", utc=True), name="Timestamp")
    source_columns = tuple(x.decode("utf-8") if isinstance(x, bytes) else str(x)
                           for x in ds["axis0"][:])
    measurement_shape = ds["block0_values"].shape   # metadata, not values
assert measurement_shape == (len(full_index), len(source_columns))
assert len(full_index) == 131617
assert full_index[0] == TRAIN_START
assert full_index[-1] == pd.Timestamp("2019-09-01 00:00:00", tz="UTC")
assert full_index.is_monotonic_increasing and full_index.is_unique
assert (full_index[1:] - full_index[:-1] == pd.Timedelta(minutes=5)).all()
required_raw = (set(F0_HISTORY_SOURCES.values()) - {
    "Humidity_clean", "Pressure_clean[mbar]", "wind_direction_sin", "wind_direction_cos"
}) | {TARGET_COLUMN, "Pressure[mbar]", "HUMIDITY[%]", "WIND_DIR[deg]"}
assert required_raw.issubset(source_columns), f"Missing raw columns: {sorted(required_raw-set(source_columns))}"

n_development = int(full_index.searchsorted(CUTOFF, side="left"))
assert n_development == 105108
assert full_index[n_development-1] < CUTOFF <= full_index[n_development]
refit_issue_times = {}
for horizon, steps in FORECAST_HORIZONS.items():
    issues = full_index[DAILY_LAG_STEPS:n_development-steps]
    valid_time = pd.Series(issues + pd.Timedelta(minutes=horizon), index=issues, name="valid_time")
    assert make_terminal_refit_mask(valid_time).all()
    assert len(issues) == EXPECTED_TERMINAL_REFIT_COUNTS_BY_HORIZON[horizon]
    assert len(issues) == selections[horizon]["refit"]["training_rows_expected"]
    assert issues[0] == pd.Timestamp("2018-06-02 00:00:00", tz="UTC")
    assert valid_time.iloc[-1] == CUTOFF - pd.Timedelta(minutes=5)
    refit_issue_times[horizon] = issues
print("TIMELINE METADATA AUDIT: PASSED — 131617 raw timestamps, 105108 pre-cutoff")
print("DEVELOPMENT REFIT POPULATIONS: 104817 / 104814 / 104808")
print("HDF5 measurement values read so far: ZERO")

TIMELINE METADATA AUDIT: PASSED — 131617 raw timestamps, 105108 pre-cutoff
DEVELOPMENT REFIT POPULATIONS: 104817 / 104814 / 104808
HDF5 measurement values read so far: ZERO


## 4. Rebuild model-ready F1/F2 inputs from **pre-cutoff measurements only**

The only raw measurement read is the explicit HDF5 hyperslab `[:n_development, :]`. Features match Notebook 10/13 in sequence, names and valid-time alignment: causal 118-column F0 → 15 `valid_` Gregorian columns → five `valid_` solar columns for F2 (with the frozen +2-minute reference convention). No learned preprocessing, F3/F4, terminal targets, or target-dependent feature engineering is added.

The function checks each horizon independently and computes an ordered feature-schema and numerical training-input SHA-256. In read-only mode we discard completed matrices promptly; in explicitly authorized mode only the requested horizon remains in memory.

In [4]:
from src.preprocessing import build_cleaned_solete_features, build_gregorian_features
from src.forecasting_protocol import build_f0_history, build_forecast_targets
from src.astronomy_features import (
    SOLETE_ASTRONOMY_OFFSET, SOLETE_SOLAR_ALTITUDE_CONVENTION_M,
    build_solar_f2_features,
)

assert SOLETE_ASTRONOMY_OFFSET == pd.Timedelta(minutes=2)
assert SOLETE_SOLAR_ALTITUDE_CONVENTION_M == 0.0
with h5py.File(H5_PATH, "r") as f:
    # A single explicit row slice; NEVER use f['DATA/block0_values'][:] here.
    development_values = f["DATA/block0_values"][:n_development, :]
assert development_values.shape == (n_development, len(source_columns))
raw_development = pd.DataFrame(development_values, index=full_index[:n_development],
                               columns=source_columns)
del development_values
assert raw_development.index[-1] < CUTOFF
assert not (raw_development.index >= CUTOFF).any()
clean_development = build_cleaned_solete_features(raw_development)
f0_development = build_f0_history(clean_development)
targets_development = build_forecast_targets(clean_development)
assert f0_development.shape[1] == 118
print("DEVELOPMENT-ONLY HDF5 MEASUREMENT ROWS LOADED:", len(raw_development))
print("Terminal-period measurement rows accessed: ZERO")

def build_locked_design(horizon: int):
    """Return the raw-frozen design and label arrays; never access terminal data."""
    assert horizon in (15, 30, 60)
    spec = selections[horizon]
    label = targets_development[horizon]
    eligible = (f0_development.notna().all(axis=1)
                & label["target_P_Solar_kW"].notna()
                & label["valid_time"].lt(CUTOFF))
    issues = clean_development.index[eligible]
    assert issues.equals(refit_issue_times[horizon])
    valid_times = pd.DatetimeIndex(label.loc[issues, "valid_time"], name="valid_time")
    assert valid_times.max() < CUTOFF
    assert valid_times.isin(raw_development.index).all()
    assert (issues + pd.Timedelta(minutes=horizon) == valid_times).all()
    y = label.loc[issues, "target_P_Solar_kW"].copy()
    f0 = f0_development.loc[issues].rename(columns={
        "P_Solar_kw_lag_288": "P_Solar_kw_issue_minus_24h"
    })
    f1_inc = build_gregorian_features(valid_times).set_axis(issues).add_prefix("valid_")
    X = pd.concat([f0, f1_inc], axis=1)
    if spec["feature_family"] == "F2":
        f2_inc = build_solar_f2_features(valid_times).set_axis(issues).add_prefix("valid_")
        assert f2_inc.shape[1] == 5
        X = pd.concat([X, f2_inc], axis=1)
    else:
        assert spec["feature_family"] == "F1"
    assert X.index.equals(y.index) and X.columns.is_unique
    assert X.index.equals(refit_issue_times[horizon])
    assert X.shape == (spec["refit"]["training_rows_expected"],
                       {"F1": 133, "F2": 138}[spec["feature_family"]])
    assert np.isfinite(X.to_numpy(dtype=np.float64)).all()
    assert np.isfinite(y.to_numpy(dtype=np.float64)).all()
    assert X.columns[:118].tolist() == f0.columns.tolist()
    assert X.columns[118:133].tolist() == f1_inc.columns.tolist()
    assert valid_times.max() == CUTOFF - pd.Timedelta(minutes=5)
    return X, y, valid_times

def fingerprints_of_design(X, y, valid_times):
    """Stable typed/ordered digest of feature schema, timestamp and numeric training content."""
    cols = list(X.columns)
    schema_sha = hashlib.sha256(json.dumps(cols, ensure_ascii=False).encode("utf-8")).hexdigest()
    digest = hashlib.sha256()
    digest.update(b"SOLETE_REVISED_REFIT_INPUT_V1\0")
    digest.update(bytes.fromhex(schema_sha))
    digest.update(np.asarray(X.index.asi8, dtype="<i8").tobytes())
    digest.update(np.asarray(valid_times.asi8, dtype="<i8").tobytes())
    # Hash in blocks rather than allocating a second full-size design matrix.
    for start in range(0, len(X), 2048):
        block = np.asarray(X.iloc[start:start+2048].to_numpy(dtype=np.float64), dtype="<f8", order="C")
        digest.update(block.tobytes(order="C"))
    digest.update(np.asarray(y.to_numpy(dtype=np.float64), dtype="<f8").tobytes())
    return schema_sha, digest.hexdigest()

read_only_input_summary = []
selected_payload = None
for horizon in (15, 30, 60):
    X, y, valid_times = build_locked_design(horizon)
    schema_sha, training_sha = fingerprints_of_design(X, y, valid_times)
    read_only_input_summary.append({
        "horizon_minutes": horizon,
        "model": "LightGBM_C1",
        "feature_family": selections[horizon]["feature_family"],
        "training_rows": len(X),
        "feature_columns": X.shape[1],
        "last_valid_time_utc": valid_times[-1].isoformat(),
        "feature_schema_sha256": schema_sha,
        "training_input_sha256": training_sha,
    })
    if RUN_FINAL_REFIT and horizon in REFIT_HORIZONS:
        selected_payload = (horizon, X, y, valid_times, schema_sha, training_sha)
    else:
        del X, y, valid_times

input_summary = pd.DataFrame(read_only_input_summary)
assert input_summary["training_rows"].tolist() == [104817, 104814, 104808]
assert input_summary["feature_columns"].tolist() == [133, 138, 138]
assert len(input_summary) == 3
assert selected_payload is not None if RUN_FINAL_REFIT else selected_payload is None
display(input_summary)
print("FROZEN DEVELOPMENT INPUT RECONSTRUCTION: PASSED")
print("Training: NOT YET PERFORMED | Terminal measurements/targets: ZERO")
# Training only consumes selected_payload. Drop shared raw measurement arrays now.
del raw_development, clean_development, f0_development, targets_development

DEVELOPMENT-ONLY HDF5 MEASUREMENT ROWS LOADED: 105108
Terminal-period measurement rows accessed: ZERO


,horizon_minutes,model,feature_family,training_rows,feature_columns,last_valid_time_utc,feature_schema_sha256,training_input_sha256
0,15,LightGBM_C1,F1,104817,133,2019-05-31T22:55:00+00:00,abacbd22f6a80e771a56ee5a0c18d74e860a5775c990e4...,3126204497bd8f43ddcf30bc39a175ec593021e600fb42...
1,30,LightGBM_C1,F2,104814,138,2019-05-31T22:55:00+00:00,6cfaaebb47ace957a7d016d8e6fd733b391bda2b0ccf82...,192df78680951e88cd29ffb2211f5fc2a207f9e5c9599f...
2,60,LightGBM_C1,F2,104808,138,2019-05-31T22:55:00+00:00,6cfaaebb47ace957a7d016d8e6fd733b391bda2b0ccf82...,87165539ff100ff27684caed0f3ed35b57414c51b3e541...


FROZEN DEVELOPMENT INPUT RECONSTRUCTION: PASSED
Training: NOT YET PERFORMED | Terminal measurements/targets: ZERO


## 5. Read-only audit of any previously saved final refits

The new model directory is separate from archived CV results. A completed horizon must have **both** a native LightGBM `.txt` model and a matching `.json` provenance record. Refuse incomplete pairs, modified input hashes, wrong source lock, or changed model artifacts. Inspection never reads model inputs from the terminal period and does not load a model for prediction.

In [5]:
MODEL_DIR = PROJECT_ROOT / "results/final_model_refits/revised_lightgbm_v1"

def model_paths(horizon):
    assert horizon in (15, 30, 60)
    stem = f"lightgbm_c1_h{horizon}_{selections[horizon]['feature_family']}"
    return MODEL_DIR / f"{stem}.txt", MODEL_DIR / f"{stem}.json"

def verify_existing_refit(horizon, expected_input_row):
    model_path, record_path = model_paths(horizon)
    found = (model_path.is_file(), record_path.is_file())
    if not any(found):
        return "NOT_FITTED", None
    if not all(found):
        raise RuntimeError(f"Incomplete refit pair; manual investigation required: {model_path}")
    record = json.loads(record_path.read_text(encoding="utf-8"))
    spec = selections[horizon]
    expected_identity = {
        "horizon_minutes": horizon,
        "model": "LightGBM",
        "candidate_id": "LightGBM_C1",
        "feature_family": spec["feature_family"],
        "configuration": dict(spec["configuration"]),
        "fixed_parameters": dict(LIGHTGBM_FIXED_PARAMS),
        "training_rows": spec["refit"]["training_rows_expected"],
        "feature_columns": expected_input_row["feature_columns"],
        "feature_schema_sha256": expected_input_row["feature_schema_sha256"],
        "training_input_sha256": expected_input_row["training_input_sha256"],
        "lock_commit": LOCK_COMMIT,
        "lock_sha256": lock_sha256,
        "last_label_valid_time_utc": expected_input_row["last_valid_time_utc"],
    }
    assert record.get("schema_version") == 1
    assert record.get("kind") == "SOLETE_REVISED_FINAL_DEVELOPMENT_LIGHTGBM_REFIT"
    assert record.get("terminal_evaluated") is False
    assert record.get("predictions_generated") is False
    assert record.get("early_stopping_used") is False
    assert record.get("model_filename") == model_path.name
    assert record.get("training_identity") == expected_identity, (
        f"Refit training identity changed for {horizon} minutes"
    )
    assert record.get("dependencies") == lock["reproducibility"]["dependencies_pinned_from_pyproject_toml"]
    assert record.get("model_size_bytes") == model_path.stat().st_size
    actual_sha = hashlib.sha256(model_path.read_bytes()).hexdigest()
    assert record.get("model_sha256") == actual_sha, f"Model file modified: {model_path}"
    assert record.get("fitted_boosting_iterations") == 300
    return "VERIFIED_EXISTING", record

status_by_horizon = {}
for row in read_only_input_summary:
    h = row["horizon_minutes"]
    status, prior = verify_existing_refit(h, row)
    status_by_horizon[h] = status
    print(f"{h} min: {status}")
print("COMPLETED REFIT CHECKPOINT AUDIT: PASSED (no writes)")

15 min: NOT_FITTED
30 min: NOT_FITTED
60 min: NOT_FITTED
COMPLETED REFIT CHECKPOINT AUDIT: PASSED (no writes)


## 6. **Opt-in only** — single-horizon final fit and native model archival

**Do not enable before reviewing the read-only run.** After explicit separate authorization, edit Section 1 in a working copy to set `RUN_FINAL_REFIT=True`, exactly one of `(15,)`/`(30,)`/`(60,)`, and the full token. Restart Kernel → Run All. The notebook fits **one** selected model on its **entire eligible development-only population** with the frozen LightGBM factory and **300 fixed rounds**. No early stopping, validation set, `.predict()`, terminal labels or scoring is allowed.

Checkpoint writes (when authorized) use a new directory and separate native `.txt` model + JSON provenance. Existing valid pairs are **reused without refitting**; corrupt/partial pairs fail closed. Output files are installed without overwriting existing files. A crash between the two files creates an incomplete pair that needs investigation, not automatic reset.

**Never commit the notebook with the opt-in switches enabled.** The model binary should be kept out of ordinary Git until storage/provenance policy is decided.

In [6]:
import tempfile

if not RUN_FINAL_REFIT:
    print("FINAL DEVELOPMENT REFIT: DISABLED — Restart/Run All is read-only")
    assert REFIT_HORIZONS == () and selected_payload is None
else:
    assert REFIT_AUTHORIZATION_TOKEN == "AUTHORIZE_FROZEN_DEVELOPMENT_REFIT"
    assert len(REFIT_HORIZONS) == 1
    horizon = REFIT_HORIZONS[0]
    assert selected_payload is not None and selected_payload[0] == horizon
    row = next(r for r in read_only_input_summary if r["horizon_minutes"] == horizon)
    prior_status, _ = verify_existing_refit(horizon, row)
    if prior_status == "VERIFIED_EXISTING":
        print(f"{horizon} min: verified completed refit reused — zero new fits")
    else:
        assert prior_status == "NOT_FITTED"
        X, y, valid_times, schema_sha, training_sha = selected_payload[1:]
        assert len(X) == selections[horizon]["refit"]["training_rows_expected"]
        assert schema_sha == row["feature_schema_sha256"]
        assert training_sha == row["training_input_sha256"]
        assert valid_times.max() < CUTOFF
        # Artifacts are outside frozen 'results/development_only'.
        MODEL_DIR.mkdir(parents=True, exist_ok=True)
        model_path, record_path = model_paths(horizon)
        lock_path_horizon = MODEL_DIR / (model_path.stem + ".refit.lock")
        try:
            lock_fd = os.open(lock_path_horizon, os.O_CREAT | os.O_EXCL | os.O_WRONLY, 0o600)
        except FileExistsError as e:
            raise RuntimeError(f"Another refit or stale lock requires review: {lock_path_horizon}") from e
        os.close(lock_fd)
        temp_paths = []
        try:
            # Recheck under per-horizon run lock; refuse partial or conflicting files.
            assert verify_existing_refit(horizon, row)[0] == "NOT_FITTED"
            estimator = build_lightgbm_model(selections[horizon]["configuration"])
            assert not hasattr(estimator, "booster_")
            started = time.perf_counter()
            estimator.fit(X, y)   # ONLY permitted training call in this notebook.
            elapsed_seconds = time.perf_counter() - started
            booster = estimator.booster_
            assert booster.current_iteration() == 300
            assert int(estimator.n_features_in_) == X.shape[1]
            assert list(estimator.feature_name_) == list(X.columns)
            fd, tmp_name = tempfile.mkstemp(prefix=".refit-model-", suffix=".txt", dir=MODEL_DIR)
            os.close(fd)
            tmp_path = Path(tmp_name)
            temp_paths.append(tmp_path)
            booster.save_model(str(tmp_path), num_iteration=-1)
            model_digest = hashlib.sha256(tmp_path.read_bytes()).hexdigest()
            assert tmp_path.stat().st_size > 0
            identity = {
                "horizon_minutes": horizon,
                "model": "LightGBM", "candidate_id": "LightGBM_C1",
                "feature_family": selections[horizon]["feature_family"],
                "configuration": dict(selections[horizon]["configuration"]),
                "fixed_parameters": dict(LIGHTGBM_FIXED_PARAMS),
                "training_rows": int(len(X)),
                "feature_columns": int(X.shape[1]),
                "feature_schema_sha256": schema_sha,
                "training_input_sha256": training_sha,
                "lock_commit": LOCK_COMMIT,
                "lock_sha256": lock_sha256,
                "last_label_valid_time_utc": row["last_valid_time_utc"],
            }
            record = {
                "schema_version": 1,
                "kind": "SOLETE_REVISED_FINAL_DEVELOPMENT_LIGHTGBM_REFIT",
                "training_identity": identity,
                "terminal_evaluated": False,
                "predictions_generated": False,
                "early_stopping_used": False,
                "model_filename": model_path.name,
                "model_sha256": model_digest,
                "model_size_bytes": tmp_path.stat().st_size,
                "fitted_boosting_iterations": int(booster.current_iteration()),
                "fit_elapsed_seconds": elapsed_seconds,
                "dependencies": lock["reproducibility"]["dependencies_pinned_from_pyproject_toml"],
                "python_version": platform.python_version(),
                "platform": platform.platform(),
                "fitted_on_checkout_head": git("rev-parse", "HEAD").stdout.strip(),
            }
            fd, temp_record_name = tempfile.mkstemp(prefix=".refit-record-", suffix=".json", dir=MODEL_DIR)
            record_tmp_path = Path(temp_record_name)
            temp_paths.append(record_tmp_path)
            with os.fdopen(fd, "w", encoding="utf-8") as f:
                json.dump(record, f, sort_keys=True, indent=2, allow_nan=False)
                f.write("\n")
                f.flush()
                os.fsync(f.fileno())
            # Hard-link install is atomic and does NOT overwrite a preexisting file.
            os.link(tmp_path, model_path)
            os.link(record_tmp_path, record_path)
            assert verify_existing_refit(horizon, row)[0] == "VERIFIED_EXISTING"
            print(f"{horizon} min: final development-only model refit and archived")
            print(f"Model SHA-256: {model_digest}")
            print("Terminal forecasts, scores and held-out measurement reads: ZERO")
        finally:
            for temp_path in temp_paths:
                temp_path.unlink(missing_ok=True)
            lock_path_horizon.unlink(missing_ok=True)
    del selected_payload
    selected_payload = None

FINAL DEVELOPMENT REFIT: DISABLED — Restart/Run All is read-only


## 7. Final execution report and downstream boundary

A green read-only report means **inputs and preconditions are ready**, not that any final model has been trained. A future refit must be separately and deliberately authorized, **one horizon at a time**. Even after refit, this notebook creates **zero forecasts** and makes **zero claims of independent test performance**.

The historical June–August 2019 period has prior exposure from legacy research. Later evaluation would be a separately authorized retrospective analysis, not a genuinely untouched test. For independent confirmation, obtain genuinely uninspected later observations and pre-register the scoring process. F3→F4 Panchang mechanism experiments should remain a distinct research stage.

In [7]:
assert TERMINAL_EVALUATION_AUTHORIZED is False
assert len(input_summary) == 3
assert input_summary["training_rows"].tolist() == [104817, 104814, 104808]
assert input_summary["feature_columns"].tolist() == [133, 138, 138]
if not RUN_FINAL_REFIT:
    assert all(status_by_horizon[h] in ("NOT_FITTED", "VERIFIED_EXISTING") for h in (15, 30, 60))
    print("NOTEBOOK 14 READ-ONLY FINAL-REFIT PREFLIGHT: PASSED")
    print("Model training by this notebook execution: ZERO")
else:
    for h in REFIT_HORIZONS:
        row = next(r for r in read_only_input_summary if r["horizon_minutes"] == h)
        assert verify_existing_refit(h, row)[0] == "VERIFIED_EXISTING"
    print("NOTEBOOK 14 EXPLICITLY AUTHORIZED SINGLE-HORIZON REFIT: VERIFIED")
print("Frozen model choices: LightGBM F1 / F2 / F2")
print("Development-only refit populations: 104817 / 104814 / 104808")
print("TERMINAL DATA ACCESS: ZERO | TERMINAL FORECASTS: ZERO | TERMINAL EVALUATION: DISABLED")

NOTEBOOK 14 READ-ONLY FINAL-REFIT PREFLIGHT: PASSED
Model training by this notebook execution: ZERO
Frozen model choices: LightGBM F1 / F2 / F2
Development-only refit populations: 104817 / 104814 / 104808
TERMINAL DATA ACCESS: ZERO | TERMINAL FORECASTS: ZERO | TERMINAL EVALUATION: DISABLED


## Next step

1. Run the notebook **as saved**, with `RUN_FINAL_REFIT=False`; inspect that all preflight cells succeed and upload the executed notebook for an independent check.
2. After the read-only audit is approved, decide separately whether to authorize one final model refit. If enabled, save a **working copy** with activation flags and revert flags in the research copy before committing.
3. Do **not** use the older historical Notebook 14 as a substitute for a clean cross-dataset or terminal evaluation: its historical conclusions follow a different methodology.

This notebook must not edit the selection lock, frozen model code, completed Notebooks 10–13, or any archived development-CV checkpoint.